# Incremental

The incremental materialisation strategy assumes that at first run dbt materialises model in the database and then updates it accroding to a specific logic.

The logic is determined by the **incremental strategy**. This section considers the different incremental strategies, their configurations, and underlying processes.

## Append

The append strategy just adds new records to the already materialised model.

## Delete+insert

The `delete+insert` strategy replaces the existing data with the new query records. The columns defined in the unique key are used to identify the records that have to be replaced.

---

Consider the example that shows the main idea of the `delete+insert` strategy:

In [13]:
# init
dbt init delete_insert --profile knowledge -q
cd delete_insert

In [14]:
# file delete_insert/seeds/data.csv
id,value
1,"value1"
2,"value2"

The model that follows `delete+insert` incremental strategy:

In [17]:
# file delete_insert/models/experimental.sql
{{
    config(
        materialized = 'incremental',
        incremental_strategy = 'delete+insert',
        unique_key = ['id']
    )
}}

select *
from {{ ref('data') }} as source_data

The `unieque_key = ['id']` column is used to identify new records.

The intial run of the model:

In [18]:
dbt seed -q --full-refresh
dbt run -q --full-refresh --select experimental
dbt show -q --select experimental

| id | value  |
| -- | ------ |
|  1 | value1 |
|  2 | value2 |



Data update:

In [19]:
# file delete_insert/seeds/data.csv
id,value
1,"value1"
2,"new value"
3,"value3"

In [22]:
dbt seed -q --full-refresh
dbt run -q --select experimental
dbt show -q --select experimental

| id | value     |
| -- | --------- |
|  1 | value1    |
|  2 | new value |
|  3 | value3    |



Note that the value under `id=2` is updated according to the incremental run.